# Richmond, VA — Land Value Tax Model

**City real-estate levy, revenue-neutral 4:1 split-rate, all current exemptions preserved.**

Richmond is one of Virginia's **independent cities** — it belongs to no county and is its own
county-equivalent for tax and Census purposes (FIPS 51760). There is therefore **no county overlay**:
the city's general real-estate tax *is* the whole local real-property levy. Virginia assesses all real
property at **100% of fair market value** (Va. Code § 58.1-3201); there is no assessment ratio or
equalization factor, so the current tax is simply `assessed value × rate`. This model holds Richmond's
rate of **$1.20 per $100** (12.0 mills) — unchanged for 19 consecutive years — and restructures the city
real-estate levy into a revenue-neutral **4:1 split-rate** (land assessed value taxed 4× improvement
assessed value). Virginia's uniformity clause (Const. Art. X § 1) currently bars a true split-rate
without enabling authority, so this is an illustrative revenue-neutral restructuring of the existing base.

| | |
|---|---|
| Geographic scope | City of Richmond (independent city), FIPS 51760 |
| Levy modeled | City real-estate tax only, single citywide rate on 100% FMV — no county overlay |
| Reform | Revenue-neutral 4:1 split-rate (land taxed 4× improvements) |
| Assessment | 100% of fair market value (Va. Code § 58.1-3201); no ratio |
| Rate | **$1.20 per $100 = 12.0 mills** — FY2027 Adopted Budget, "Tax Rates" appendix: *"Real Estate / $1.20 per $100 of Assessed Value : 2008 - 2026"*; also rva.gov/finance/real-estate |
| Revenue control | FY2027 Adopted Budget, *General Fund Revenues: Summary by Category* — **Real Property Taxes – Current = $510,500,860** |
| Base control | FY2027 Adopted Budget Table 7 (*Assessed Value of Taxable Property Less Tax Exempt Property*), Jan 1 2025 roll: real property total **$52,992,485,155**, exempt **$11,419,995,000** |
| Assessment roll | **January 1, 2026** (`AssessmentDate` on 76,554 of 76,879 parcels) |
| Data source | City of Richmond ArcGIS `Parcels` FeatureServer (assessor parcel base: land/improvement values, property class, land use, exempt code, owner) + `All_Address_Parcel_Asr_View` for site addresses |

**Sources**
- FY2027 Adopted Annual Fiscal Plan: <https://rva.gov/sites/default/files/2026-07/Adopted%20FY%202027%20Annual%20Budget%20and%20FY%202027-2031%20CIP%20Plan.pdf> (rate p. 584 & p. 67; revenue table p. 45; Table 7 p. 531)
- Dept. of Finance, Real Estate: <https://www.rva.gov/finance/real-estate>
- Parcels: <https://services1.arcgis.com/k3vhq11XkBNeeOfM/ArcGIS/rest/services/Parcels/FeatureServer/0>

**Three jurisdiction-specific data treatments (documented in Sections 3, 4 and 5b):**
- **Exemptions come from the assessor's own `TaxExemptCode`.** Any non-null code is held out as fully
  exempt. The resulting exempt total reconciles to the budget's published exempt figure (Section 4).
- **Public service corporations dropped.** Exempt codes `900`/`950` (Dominion/Virginia Electric & Power,
  CSX, Norfolk Southern, Columbia Gas, telecoms) are assessed by the Virginia State Corporation
  Commission, not the local assessor. The budget books them on a **separate** `PSC Property Taxes` line,
  so they are outside the levy this model reproduces and are dropped explicitly.
- **Partial exemptions cannot be split.** Exempt code `620` (411 parcels, $1.52B) is Richmond's
  rehabilitated-real-estate abatement / affordable-housing population — a *partial* exemption of the
  post-rehab value increment. The parcel layer carries no abated-amount field, so these are held out in
  full. See the limitation note in Section 5b.

## Section 1 — Imports and Setup

In [ ]:
import sys
import io
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)

# Constants
CITY_NAME = 'richmond'
STATE_FIPS = '51'                 # Virginia
COUNTY_FIPS = '760'               # Richmond city (independent city = county-equivalent 51760)
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0

CITY_MILLAGE = 12.0               # $1.20 per $100 of assessed value (unchanged 2008-2026)

# FY2027 Adopted Budget, "General Fund Revenues: Summary by Category":
#   Real Property Taxes - Current .......... 510,500,860   (FY2027 Adopted)
# This is a *net* budgeted figure: the budget assumes a 97% collection rate and books the
# rehabilitation-tax-abatement credit as "a reduction to the source". The modeled figure below
# is the GROSS certified levy (base x rate), so a small positive gap is expected and explained.
OFFICIAL_REVENUE = 510_500_860
ASSUMED_COLLECTION_RATE = 0.97    # FY2027 budget: "an anticipated 97 percent collection rate"

# FY2027 Adopted Budget, Table 7 (Jan 1 2025 roll) — the base control, one roll year behind our data.
OFFICIAL_TOTAL_AV_2025 = 52_992_485_155
OFFICIAL_EXEMPT_AV_2025 = 11_419_995_000

# Parcel-map export (see lvt/parcel_map.py) — per-parcel identity columns carried onto the
# GeoParquet/interactive map but NOT the flat cross-city CSV.
PARCEL_ID_COL = 'PIN'
OWNER_NAME_COL = 'OwnerName'
OWNER_ADDRESS_COL = 'SiteAddress'
# Richmond retired its own property-search app in favour of actDataScout
# (https://www.actdatascout.com/RealProperty/Virginia/Richmond), which is behind bot protection
# and exposes no verifiable stable per-parcel deep link. Per build-notebook.md, emit no link
# rather than one that 404s.
PARCEL_URL_TEMPLATE = None

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print(f"Rate: ${CITY_MILLAGE/10:.2f} per $100 = {CITY_MILLAGE} mills (100% FMV, no assessment ratio)")
print(f"Split ratio: {LAND_IMPROVEMENT_RATIO:.0f}:1 (land:improvement)")

## Section 2 — Load Parcel Data

Parcels come from the City of Richmond's ArcGIS **`Parcels` FeatureServer** (the assessor parcel base
published by the City Assessor's office). The layer carries current land and improvement values,
property class, land use, the assessor's tax-exempt code, owner and geometry — everything the model
needs. Geometry is requested in EPSG:4326 (`outSR=4326`) so the Census centroid join works directly.

Site addresses are not on the parcel layer, so they are joined from the city's
`All_Address_Parcel_Asr_View` (address-level view, one row per address). It is deduplicated to one
primary address per PIN and used only for the interactive map / downstream example scouting — never in
the model.

| Concept | Column | Notes |
|---|---|---|
| Parcel ID | `PIN` | Assessor parcel identification number; the unique key (76,878 distinct of 76,879 rows) |
| Map parcel | `ParcelID` | Ground parcel; multiple PINs (condo units) can share one `ParcelID` |
| Current land value | `LandValue` | Fair market value (100% FMV) |
| Current improvement value | `DwellingValue` | Buildings/structures at 100% FMV (name is a misnomer — it covers commercial and industrial improvements too) |
| Total value | `TotalValue` | Land + improvement |
| Property class | `PropertyClassID` / `PropertyClass` | 3-digit assessor class, 118 distinct values — the classification authority |
| Land use | `LandUse` | Coarse rollup (Single Family / Multi-Family / Commercial / …), used only as a cross-check |
| Exempt code | `TaxExemptCode` | Non-null ⇒ on the assessor's exempt roll |
| Land area | `LandSqFt` | |
| Owner | `OwnerName` | Used to identify public service corporations |

**Value nulls are real.** `DwellingValue` is null (not zero) on 6,902 parcels and `LandValue` on 291 —
the assessor writes no row rather than a zero. They are coerced to 0 and clipped, which is what makes
the vacant-land override below fire correctly.

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
ADDR_PATH = DATA_DIR / 'site_addresses.parquet'

ARCGIS = 'https://services1.arcgis.com/k3vhq11XkBNeeOfM/ArcGIS/rest/services'
PARCEL_LAYER = f'{ARCGIS}/Parcels/FeatureServer/0/query'
ADDR_LAYER = f'{ARCGIS}/All_Address_Parcel_Asr_View/FeatureServer/3/query'
FIELDS = ('ParcelID,PIN,CountOfPIN,OwnerName,AsrLocationBldgNo,MailAddress,MailCity,MailState,'
          'MailZip,AssessmentDate,LandValue,DwellingValue,TotalValue,LandSqFt,ProvalAsmtNhood,'
          'TaxExemptCode,PropertyClassID,PropertyClass,LandUse')


def fetch_richmond_parcels():
    """Paginate the Richmond Parcels FeatureServer as GeoJSON in EPSG:4326."""
    total = requests.get(PARCEL_LAYER, params={
        'f': 'json', 'where': '1=1', 'returnCountOnly': 'true'}, timeout=60).json()['count']
    parts, offset = [], 0
    while offset < total:
        r = requests.get(PARCEL_LAYER, params={
            'f': 'geojson', 'where': '1=1', 'outFields': FIELDS, 'returnGeometry': 'true',
            'outSR': 4326, 'resultOffset': offset, 'resultRecordCount': 2000,
            'orderByFields': 'ParcelID'}, timeout=300)
        r.raise_for_status()
        g = gpd.read_file(io.StringIO(r.text))
        if len(g) == 0:
            break
        parts.append(g)
        offset += len(g)
        time.sleep(0.1)
    out = pd.concat(parts, ignore_index=True)
    return gpd.GeoDataFrame(out, geometry='geometry', crs='EPSG:4326')


def fetch_richmond_site_addresses():
    """One primary site address per PIN from the city's address/parcel/assessor view."""
    total = requests.get(ADDR_LAYER, params={
        'f': 'json', 'where': '1=1', 'returnCountOnly': 'true'}, timeout=60).json()['count']
    rows, offset = [], 0
    while offset < total:
        r = requests.get(ADDR_LAYER, params={
            'f': 'json', 'where': '1=1', 'outFields': 'PIN,AddressLabelWithUnit',
            'returnGeometry': 'false', 'resultOffset': offset, 'resultRecordCount': 2000,
            'orderByFields': 'OBJECTID'}, timeout=180)
        r.raise_for_status()
        feats = r.json().get('features', [])
        if not feats:
            break
        rows += [f['attributes'] for f in feats]
        offset += len(feats)
        time.sleep(0.05)
    a = pd.DataFrame(rows).dropna(subset=['PIN'])
    a['AddressLabelWithUnit'] = a['AddressLabelWithUnit'].astype(str).str.strip()
    a = a[a['AddressLabelWithUnit'].str.len() > 0]
    a = a.sort_values(['PIN', 'AddressLabelWithUnit']).drop_duplicates('PIN', keep='first')
    return a.rename(columns={'AddressLabelWithUnit': 'SiteAddress'})


if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    gdf = fetch_richmond_parcels()
    gdf.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(gdf):,} parcels")

if ADDR_PATH.exists():
    addr = pd.read_parquet(ADDR_PATH)
else:
    addr = fetch_richmond_site_addresses()
    addr.to_parquet(ADDR_PATH, index=False)

gdf['PIN'] = gdf['PIN'].astype(str)
addr['PIN'] = addr['PIN'].astype(str)
gdf = gdf.merge(addr[['PIN', 'SiteAddress']], on='PIN', how='left')
print(f"Site address matched on {gdf['SiteAddress'].notna().mean()*100:.1f}% of parcels")

# Values: the assessor writes NULL (not 0) where there is nothing to book.
for c in ['LandValue', 'DwellingValue', 'TotalValue']:
    gdf[c] = pd.to_numeric(gdf[c], errors='coerce').fillna(0).clip(lower=0)
gdf['PropertyClassID'] = gdf['PropertyClassID'].fillna('').astype(str).str.strip()
gdf['taxable_land_value'] = gdf['LandValue']
gdf['taxable_improvement_value'] = gdf['DwellingValue']
gdf['total_value'] = gdf['LandValue'] + gdf['DwellingValue']

# AssessmentDate arrives from ArcGIS as epoch milliseconds.
_roll = pd.to_datetime(gdf['AssessmentDate'], unit='ms', errors='coerce').dt.date
_modal = _roll.mode().iloc[0]
print(f"Assessment roll: {_modal} on {int((_roll == _modal).sum()):,} of {len(gdf):,} parcels")
print(f"Total assessed value (all parcels): ${gdf['total_value'].sum():,.0f}")
print(f"  land:         ${gdf['taxable_land_value'].sum():,.0f}")
print(f"  improvements: ${gdf['taxable_improvement_value'].sum():,.0f}")
print(f"Median total value: ${gdf['total_value'].median():,.0f}")
print(f"Parcels with $0 improvement (vacant candidates): {(gdf['DwellingValue'] <= 0).sum():,}")

## Section 3 — Classify Parcels

**Fully-exempt parcels (held out).** The assessor publishes a per-parcel `TaxExemptCode`; any non-null
value puts the parcel on the exempt roll. Reading the codes against their owners identifies them:

| Code | Population |
|---|---|
| `100` `105` `106` | City of Richmond (Public Works, School Board, Recreation & Parks) |
| `150` `152` | Richmond Redevelopment & Housing Authority; Economic Development Authority |
| `200`–`221` | Commonwealth of Virginia, state agencies, VCU health, regional authorities |
| `300` | Other Virginia localities (Henrico County) |
| `350`–`406` | Federal government / USPS; churches and religious organizations |
| `450` | Cemeteries |
| `500` `510` | Private schools, colleges (University of Richmond, Trinity Episcopal) |
| `600` `601` `605` | Charitable, benevolent and supportive-housing organizations |
| `620` | **Rehabilitation abatement / affordable-housing partnerships** (partial in law — see Section 5b) |
| `630` | Maggie Walker Community Land Trust (the *land* side of CLT ground leases) |
| `698` `699` | Homeowner-association common areas |
| `711` | Owner-occupied homes on statutory relief (disabled-veteran / elderly & disabled exemption) |
| `900` `950` | **Public service corporations** — SCC-assessed, dropped entirely (see below) |

**Public service corporations (excluded).** Codes `900`/`950` cover Virginia Electric & Power
(Dominion), CSX, Norfolk Southern, Columbia Gas, and telecoms. Under Va. Code § 58.1-2600 *et seq.*
these are assessed by the **State Corporation Commission**, not the local assessor, and Richmond's
budget books them on a separate `PSC Property Taxes — Real Property` line ($5,175,740 in FY2027), not
in the `Real Property Taxes — Current` line this model reproduces. They are dropped before modeling.

**Property categories** are built from the assessor's 3-digit `PropertyClassID` — a genuinely detailed
taxonomy (118 distinct classes) that supports clean condo, small/large multi-family, mixed-use, hotel
and parking splits. Richmond's classes do **not** distinguish rowhouses from detached homes (a Fan
District rowhouse is class `120 R Two Story` like any other two-storey house), so there is no
`Townhome / Rowhouse` bucket. The standard override applies last: any parcel with $0 improvement value
becomes **Vacant Land** regardless of class.

In [ ]:
gdf['TaxExemptCode'] = gdf['TaxExemptCode'].fillna('').astype(str).str.strip()

# --- Public service corporations: SCC-assessed, outside the local real-estate levy ---
PSC_CODES = {'900', '950'}
gdf['is_psc'] = gdf['TaxExemptCode'].isin(PSC_CODES)

# --- Fully exempt: any non-null assessor exempt code (PSC handled separately) ---
gdf['full_exmp'] = ((gdf['TaxExemptCode'] != '') & ~gdf['is_psc']).astype(int)

print(f"Public service corporations (dropped): {int(gdf['is_psc'].sum()):,} "
      f"(${gdf.loc[gdf['is_psc'], 'total_value'].sum():,.0f})")
print(f"Fully-exempt parcels (held out):       {int(gdf['full_exmp'].sum()):,} "
      f"(${gdf.loc[gdf['full_exmp'] == 1, 'total_value'].sum():,.0f})")
print(f"Taxable parcels:                       {int(((gdf['full_exmp'] == 0) & ~gdf['is_psc']).sum()):,}")
print("\nTop exempt codes by value:")
_e = gdf[gdf['TaxExemptCode'] != ''].groupby('TaxExemptCode')['total_value'].agg(['size', 'sum'])
print(_e.sort_values('sum', ascending=False).head(12).to_string(
    float_format=lambda v: f"{v:,.0f}"))

VACANT = {'101', '102', '201', '301', '302', '303', '401', '501'}
PARKING = {'106', '220', '230', '306', '406', '490', '506'}
SFR = {'109', '110', '115', '120', '130', '150', '155', '196', '198', '750'}
SMALL_MF = {'160', '161', '170', '171', '180', '181'}
CONDO = {'210', '211', '212'}
LARGE_MF = {'309', '310', '315', '321', '325', '330', '345', '396'}
MIXED = {'195', '335', '341', '450'}
OTHER_RES = {'354', '355', '360', '365', '398'}
RETAIL = {'409', '410', '415', '416', '418', '419', '420', '421', '422', '425', '426', '428',
          '430', '432'}
OFFICE = {'440', '441', '442', '443', '445', '470'}
HOTEL = {'460', '461'}
INDUSTRIAL = {'509', '510', '511', '516', '532', '533', '534', '535', '536', '570', '580', '598',
              '715'}
OTHER_COMM = {'446', '447', '448', '449', '452', '454', '455', '480', '482', '491', '498'}
# Institutional (mostly exempt; the taxable remnant is small) and common-area / accessory parcels.
OTHER = {'105', '190', '191', '193', '205', '206', '305', '405', '456', '457', '458', '466',
         '467', '468', '469', '471', '472',
         # 711 'P Public Service Corp.' — every one carries an exempt code (SCC-assessed
         # utilities, or the state's Virginia Passenger Rail Authority under code 209),
         # so none survives into the model; mapped only to keep the diagnostic clean.
         '711'}

CATEGORY_BY_CLASS = {}
for _codes, _name in [
    (VACANT, 'Vacant Land'), (PARKING, 'Transportation - Parking'),
    (SFR, 'Single Family Residential'), (SMALL_MF, 'Small Multi-Family (2-4 units)'),
    (CONDO, 'Condominium'), (LARGE_MF, 'Large Multi-Family (5+ units)'),
    (MIXED, 'Mixed Use'), (OTHER_RES, 'Other Residential'),
    (RETAIL, 'Retail / General Commercial'), (OFFICE, 'Office / Commercial Condo'),
    (HOTEL, 'Hotel'), (INDUSTRIAL, 'Industrial'), (OTHER_COMM, 'Other Commercial'),
    (OTHER, 'Other'),
]:
    for _c in _codes:
        CATEGORY_BY_CLASS[_c] = _name

gdf['PROPERTY_CATEGORY'] = gdf['PropertyClassID'].map(CATEGORY_BY_CLASS).fillna('Other')
# Override: $0 improvement value -> Vacant Land, regardless of class
gdf.loc[gdf['DwellingValue'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

_unmapped = gdf.loc[~gdf['PropertyClassID'].isin(CATEGORY_BY_CLASS) & ~gdf['is_psc'], 'PropertyClassID']
print(f"\nUnmapped property classes (fall to 'Other'): {_unmapped.nunique()} "
      f"classes / {len(_unmapped):,} parcels -> {sorted(_unmapped.unique())[:12]}")
print()
print(gdf.loc[(~gdf['is_psc']) & (gdf['full_exmp'] == 0), 'PROPERTY_CATEGORY']
      .value_counts().to_string())

## Section 4 — Current Tax + Validation (Gate 1)

Virginia assesses at 100% of fair market value, so the current real-estate tax is simply
`assessed value × rate`, with no assessment ratio:

```
current_tax = (LandValue + DwellingValue) × 12.0 / 1000     (taxable parcels only)
```

**Two independent controls.**

1. **Base control.** The FY2027 Adopted Budget's Table 7 reports the **January 1, 2025** roll: real
   property total $52,992,485,155, of which $11,419,995,000 exempt. Our parcel file is the **January 1,
   2026** roll, one year later, so the check is that the year-over-year growth is coherent (Richmond's
   roll grew 5.9% from 2024 to 2025 on the same table).
2. **Revenue control.** FY2027 Adopted `Real Property Taxes – Current` = **$510,500,860**. That is a
   *net* budgeted figure — the budget assumes a **97% collection rate** and books the
   rehabilitated-real-estate abatement as *"a reduction to the source"*. The model produces the **gross
   certified levy**, so it should sit a few percent above the budget line, and should land essentially
   on the budget figure once grossed up by the collection rate.

In [ ]:
gdf['millage_rate'] = CITY_MILLAGE

# Drop public service corporations entirely (out of local levy scope)
n_psc = int(gdf['is_psc'].sum())
psc_value = float(gdf.loc[gdf['is_psc'], 'total_value'].sum())
gdf = gdf[~gdf['is_psc']].copy()

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

modeled_total_av = float(gdf['total_value'].sum())
modeled_exempt_av = float(gdf.loc[gdf['full_exmp'] == 1, 'total_value'].sum())
modeled_taxable_av = float(gdf.loc[gdf['full_exmp'] == 0, 'total_value'].sum())

print(f"Dropped {n_psc:,} public service corporation parcels (${psc_value:,.0f}).")
print()
print("Gate 1a — base control (roll years differ by one; growth must be coherent)")
print(f"  Modeled total real property (Jan 1 2026): ${modeled_total_av:,.0f}")
print(f"  Official  total real property (Jan 1 2025): ${OFFICIAL_TOTAL_AV_2025:,.0f}"
      f"   -> {(modeled_total_av/OFFICIAL_TOTAL_AV_2025-1)*100:+.1f}% YoY")
print(f"  Modeled exempt (Jan 1 2026):  ${modeled_exempt_av:,.0f}")
print(f"  Official exempt (Jan 1 2025): ${OFFICIAL_EXEMPT_AV_2025:,.0f}"
      f"   -> {(modeled_exempt_av/OFFICIAL_EXEMPT_AV_2025-1)*100:+.1f}% YoY")
print(f"  Modeled TAXABLE base (Jan 1 2026): ${modeled_taxable_av:,.0f}")
print()
print("Gate 1b — revenue control")
gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
implied_gross = OFFICIAL_REVENUE / ASSUMED_COLLECTION_RATE
gap_gross_pct = (current_revenue / implied_gross - 1) * 100
print(f"  Modeled gross levy: ${current_revenue:,.0f}   "
      f"Official (FY2027 Adopted, net): ${OFFICIAL_REVENUE:,.0f}   Gap: {gap_pct:+.2f}%")
print(f"  Budget-implied GROSS levy at {ASSUMED_COLLECTION_RATE:.0%} collection: ${implied_gross:,.0f}"
      f"   Gap: {gap_gross_pct:+.2f}%")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

## Section 5 — Split-Rate Model

Revenue-neutral 4:1 split-rate on the current land and improvement assessed values. Fully-exempt
parcels (and any carrying no assessed value at all) are held out of the solver and excluded from the
export and charts, per repo norm.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
no_value = (gdf['full_exmp'] == 0) & (gdf['total_value'] <= 0)
n_novalue = int(no_value.sum())

gdf = gdf[(gdf['full_exmp'] == 0) & (gdf['total_value'] > 0)].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt + {n_novalue:,} zero-value parcels "
      f"(excluded from model, export, charts).")
print(f"Modeled parcels: {len(gdf):,}")
print(f"Land millage: {land_millage:.4f}  (${land_millage/10:.4f} per $100)")
print(f"Improvement millage: {improvement_millage:.4f}  (${improvement_millage/10:.4f} per $100)")
print(f"  (current flat rate {CITY_MILLAGE} mills = $1.20/$100; land taxed "
      f"{LAND_IMPROVEMENT_RATIO:.0f}x improvements)")
print(f"Revenue neutrality: ${new_revenue:,.0f} vs target ${gdf['current_tax'].sum():,.0f} "
      f"(gap {(new_revenue/gdf['current_tax'].sum()-1)*100:+.4f}%)")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(
    category_summary,
    title='Richmond — 4:1 Split-Rate Tax Impact (city real-estate levy)')

## Section 5b — Read the Results (Gate 5 Artifact Scan)

Check building shares by category against economic priors, and watch for the known artifacts (Seattle
placeholder commercial improvement values, Maricopa's flat residential land ratio, St. Paul condo
token land).

In [ ]:
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / (
    d['taxable_land_value'] + d['taxable_improvement_value']).replace(0, np.nan)
summary = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
    total_value=('total_value', 'sum'),
).sort_values('median_pct', ascending=False)
print(summary.round(3).to_string())

print("\n--- Condo token-land check (St. Paul pattern) ---")
_condo = d[d['PROPERTY_CATEGORY'] == 'Condominium']
print(f"Condo parcels: {len(_condo):,}; median land value ${_condo['taxable_land_value'].median():,.0f}; "
      f"median land share {(1 - _condo['bldg_share']).median()*100:.1f}%; "
      f"at/below $1,000 land: {(_condo['taxable_land_value'] <= 1000).mean()*100:.2f}%")

print("\n--- Flat residential land-ratio check (Maricopa pattern) ---")
_sfr = d[d['PROPERTY_CATEGORY'] == 'Single Family Residential']
_ls = (1 - _sfr['bldg_share']).dropna()
print(f"SFR land share: p10 {_ls.quantile(.10)*100:.1f}% | median {_ls.median()*100:.1f}% | "
      f"p90 {_ls.quantile(.90)*100:.1f}%  (a flat assessor ratio would collapse this spread)")

print("\n--- Zero-land parcels (pay improvement millage only under the split rate) ---")
_zl = d[d['taxable_land_value'] <= 0]
print(f"{len(_zl):,} parcels, ${_zl['total_value'].sum():,.0f} of value "
      f"({_zl['total_value'].sum()/d['total_value'].sum()*100:.3f}% of the modeled base)")
print(_zl['PropertyClass'].value_counts().head(6).to_string())

### Gate 5 reading — Richmond

**No canary fires.** The scan is clean on all three known artifact patterns:

- **No placeholder improvement values.** Median building share falls in a smooth, economically
  sensible ladder — Vacant 0% → Parking 3% → Other Commercial 23% → Retail 49% → Industrial 52% →
  Office 63% → Small Multi-Family 68% → Mixed Use 71% → Single Family 73% → Condo 83% → Large
  Multi-Family 84%. Distinct categories do *not* pile up on one extreme value, and no built-up
  category sits near 0% building share (the Seattle pattern).
- **No condo collapse.** Richmond's 4,163 condo units carry real, varied land values — median land
  value $45,000, median land share 17.3%, and **0.00%** at or below a $1,000 token. No `PlatID`
  collapse is needed (the St. Paul pattern is absent).
- **No flat residential land ratio.** Single-family land share spreads 17.6% (p10) → 27.2% (median)
  → 41.4% (p90). A flat assessor ratio (the Maricopa pattern) would collapse that spread to a point.

**The one result that needs explaining: single-family homes come out roughly flat, not down.**
Median SFR change is **−0.58%** and the category pays **+0.29%** more in aggregate; 51.4% of SFR
parcels pay less and 48.6% pay more. This is not an error — it is arithmetic. Under a revenue-neutral
split rate the break-even land share is exactly the citywide land share of the tax base:

```
break-even land share = (12.0 − 6.5726) / ((26.2905 − 12.0) + (12.0 − 6.5726)) = 27.53%
citywide land share of the modeled base                                       = 27.53%
median single-family land share                                               = 27.17%
```

Richmond's typical house sits **within a third of a percentage point of the break-even line**, so the
single-family category as a whole barely moves and splits nearly 50/50 between winners and losers. The
shift instead runs from surface parking, vacant lots and low-rise commercial land toward apartments,
condos and hotels. Any deck built on this model should lead with that, not with a homeowner tax cut.

**Three documented limitations, none of which changes the direction of the result:**

1. **Partial exemptions are held out in full.** Exempt code `620` — 411 parcels, $1.52B — is
   Richmond's rehabilitated-real-estate abatement plus affordable-housing partnerships. In law that
   exempts only the *increment* of value added by rehabilitation, for a fixed term; the parcel layer
   publishes no abated-amount field, so the model holds the whole parcel out. This runs in the same
   direction as the budget, which books the abatement as *"a reduction to the source"*. Treating them
   as fully taxable instead would lift the modeled base to $45.35B — **3.4% above** the base implied
   by the budgeted revenue, i.e. a materially worse reconciliation. That is the evidence for holding
   them out.
2. **Statutory relief is only visible where the assessor stamped a code.** Virginia's elderly &
   disabled relief and the disabled-veteran exemption (Va. Code § 58.1-3219.5) show up as exempt code
   `711` on 387 owner-occupied homes. Partial relief that reduces a *bill* without changing the
   *assessment* is invisible in this file, so a small number of parcels are modeled at their full
   assessment.
3. **Community-land-trust homes carry $0 land.** Maggie Walker Community Land Trust ground leases
   split the assessment: the land sits on the trust's exempt parcels (code `630`) and the house is
   assessed to the homeowner with **zero land value** (class `750`). 72 such parcels survive into the
   model and, with 6 cell-tower parcels, make up all 78 zero-land parcels — $13.7M, 0.03% of the
   modeled base. Under a 4:1 split they pay the improvement millage only, i.e. a large cut. That is a
   real consequence of how CLT ground leases are assessed, not a data error, but it is a small and
   distinctive population worth naming rather than burying inside Single Family Residential.


## Section 6 — Exploration Chart

In [ ]:
_counts = gdf['PROPERTY_CATEGORY'].value_counts()
_keep = _counts[_counts >= 50].index
_s = (gdf[gdf['PROPERTY_CATEGORY'].isin(_keep)]
      .groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values())

fig, ax = plt.subplots(figsize=(10, 7))
_s.plot.barh(ax=ax, color=['#B7E3E0' if v < 0 else '#352977' for v in _s])
ax.axvline(0, color='#333', lw=1)
ax.set_title('Richmond — Median Tax Change % by Category (4:1 Split-Rate)')
ax.set_xlabel('Median % change')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("wrote data/category_preview.png")


## Section 7 — Census Join + Export

In [ ]:
# Census join — OPTIONAL, must happen before export
import os
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    # No key configured — the census join is optional, so skip it cleanly.
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        gdf[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                gdf = match_to_census_blockgroups(gdf, census_gdf)
                # census_gdf already carries demographics — do NOT merge census_data again.
                if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                    gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
                if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                    gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
                print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

# Standard report — PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet + interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")